# Part 2 — Cross-Domain Acne Classification (Google Colab)

**Before running:** Runtime → Change runtime type → **A100 GPU**

This notebook covers all of Part 2 end-to-end:
1. Setup (clone repo, install deps, download ACNE04)
2. Patch extraction — crop positive/negative patches from ACNE04 bounding boxes
3. Train EfficientNet-B0 classifier on ACNE04 patches
4. Download & prepare DermNet dataset
5. Evaluate on DermNet test set (Accuracy, F1, AUROC)
6. Grad-CAM visualizations on DermNet predictions
7. Reflection

Run cells top to bottom.

---
## Section 1 — Setup

In [ ]:
# Clone repo
import os

REPO_DIR = "/content/AcneDetection"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/EvxLee/AcneDetection.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print("Repo already exists — pulled latest.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install -q roboflow python-dotenv timm grad-cam scikit-learn
print("Dependencies installed.")

In [ ]:
# Set Roboflow credentials — paste your API key here
import os

os.environ["ROBOFLOW_API_KEY"]   = "YOUR_API_KEY_HERE"   # ← paste your key
os.environ["ROBOFLOW_WORKSPACE"] = "evan-lee-rrndd"
os.environ["ROBOFLOW_PROJECT"]   = "acne04-detection-p8j0d"
os.environ["ROBOFLOW_VERSION"]   = "1"

with open(f"{REPO_DIR}/.env", "w") as f:
    for k in ["ROBOFLOW_API_KEY", "ROBOFLOW_WORKSPACE", "ROBOFLOW_PROJECT", "ROBOFLOW_VERSION"]:
        f.write(f"{k}={os.environ[k]}\n")
print("Credentials set.")

In [ ]:
# Download ACNE04 dataset
!python part1_detection/roboflow_loader.py --download

In [ ]:
# Verify GPU
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
else:
    print("No GPU — go to Runtime → Change runtime type → A100")

---
## Section 2 — Patch Extraction

Creates a binary classification dataset from ACNE04 bounding boxes:

- **Positive (acne)**: crop each bounding box region, resize to 224×224
- **Negative (no_acne)**: randomly crop same-sized regions with zero overlap with any GT box

Output (standard PyTorch ImageFolder format):
```
data/patches/
├── train/
│   ├── acne/
│   └── no_acne/
└── val/
    ├── acne/
    └── no_acne/
```

In [ ]:
import json
import random
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

%matplotlib inline

DATA_DIR   = Path("data/acne04")
PATCH_DIR  = Path("data/patches")
PATCH_SIZE = 224
MIN_BOX    = 20
NEG_SIZE   = 90
SEED       = 42

SPLITS = {"train": "train", "valid": "val"}

random.seed(SEED)

In [ ]:
def is_valid_patch(patch_img, min_brightness=40, min_color_spread=15):
    """Reject dark/greyscale patches (backgrounds, hair, watermarks)."""
    arr = np.array(patch_img, dtype=float)
    if arr.mean() < min_brightness:
        return False
    channel_means = arr.mean(axis=(0, 1))
    if channel_means.max() - channel_means.min() < min_color_spread:
        return False
    return True

def has_overlap(box, gt_boxes):
    for g in gt_boxes:
        if box[0] < g[2] and box[2] > g[0] and box[1] < g[3] and box[3] > g[1]:
            return True
    return False

def sample_negative(img, img_w, img_h, gt_boxes, size, max_tries=150):
    for _ in range(max_tries):
        x1 = random.randint(0, max(0, img_w - size))
        y1 = random.randint(0, max(0, img_h - size))
        candidate = [x1, y1, x1 + size, y1 + size]
        if has_overlap(candidate, gt_boxes):
            continue
        patch = img.crop(candidate).resize((PATCH_SIZE, PATCH_SIZE), Image.BILINEAR)
        if is_valid_patch(patch):
            return candidate, patch
    return None, None

def extract_patches(acne04_split, patch_split):
    pos_dir = PATCH_DIR / patch_split / "acne"
    neg_dir = PATCH_DIR / patch_split / "no_acne"
    pos_dir.mkdir(parents=True, exist_ok=True)
    neg_dir.mkdir(parents=True, exist_ok=True)

    with open(DATA_DIR / acne04_split / "_annotations.coco.json") as f:
        coco = json.load(f)

    ann_map = {}
    for ann in coco["annotations"]:
        ann_map.setdefault(ann["image_id"], []).append(ann)

    pos_count = neg_count = skipped = rejected = 0

    for meta in coco["images"]:
        img_id = meta["id"]
        anns   = ann_map.get(img_id, [])
        if not anns:
            continue

        img      = Image.open(DATA_DIR / acne04_split / meta["file_name"]).convert("RGB")
        img_w, img_h = img.size
        gt_boxes = []

        for ann in anns:
            x, y, w, h = ann["bbox"]
            if w < MIN_BOX or h < MIN_BOX:
                skipped += 1
                continue
            x1, y1 = int(x), int(y)
            x2, y2 = min(int(x + w), img_w), min(int(y + h), img_h)
            gt_boxes.append([x1, y1, x2, y2])
            patch = img.crop((x1, y1, x2, y2)).resize((PATCH_SIZE, PATCH_SIZE), Image.BILINEAR)
            ann_id = ann["id"]
            patch.save(pos_dir / f"{img_id}_{ann_id}.jpg", quality=90)
            pos_count += 1

        for i in range(len(gt_boxes)):
            box, patch = sample_negative(img, img_w, img_h, gt_boxes, NEG_SIZE)
            if patch is None:
                rejected += 1
                continue
            patch.save(neg_dir / f"{img_id}_neg{i}.jpg", quality=90)
            neg_count += 1

    print(f"[{acne04_split}]  acne={pos_count}  no_acne={neg_count}  skipped(tiny)={skipped}  rejected(bad)={rejected}")

print("Starting patch extraction...")
for acne04_split, patch_split in SPLITS.items():
    extract_patches(acne04_split, patch_split)
print("Done.")

In [ ]:
for split in ["train", "val"]:
    for cls in ["acne", "no_acne"]:
        files = list((PATCH_DIR / split / cls).glob("*.jpg"))
        print(f"  data/patches/{split}/{cls}: {len(files)} images")

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for row, cls in enumerate(["acne", "no_acne"]):
    files = random.sample(list((PATCH_DIR / "train" / cls).glob("*.jpg")), 4)
    for col, f in enumerate(files):
        axes[row][col].imshow(Image.open(f))
        axes[row][col].set_title(cls, fontsize=9)
        axes[row][col].axis("off")
plt.suptitle("Sample patches — train set", fontsize=13)
plt.tight_layout()
plt.show()

---
## Section 3 — Train EfficientNet-B0 Classifier

Fine-tunes EfficientNet-B0 (pretrained on ImageNet) on the ACNE04 patches.
Training augmentation (color jitter, random crop, blur) helps the model generalise
to DermNet's different lighting and color profile.

**Output:** `outputs/classifier/best.pth`

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
from pathlib import Path

PATCH_DIR   = Path("data/patches")
CLF_OUT     = Path("outputs/classifier")
CLF_OUT.mkdir(parents=True, exist_ok=True)

EPOCHS      = 20
BATCH       = 64
LR          = 1e-4
NUM_WORKERS = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(PATCH_DIR / "train", transform=train_tf)
val_ds   = datasets.ImageFolder(PATCH_DIR / "val",   transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

class_names = train_ds.classes
print(f"Classes : {class_names}")
print(f"Train   : {len(train_ds)} images")
print(f"Val     : {len(val_ds)} images")

In [ ]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(model.classifier[1].in_features, 2),
)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
print("EfficientNet-B0 ready.")

In [ ]:
train_losses, val_losses, val_accs = [], [], []
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    avg_train = total_loss / len(train_loader)
    train_losses.append(avg_train)

    model.eval()
    total_loss = correct = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            total_loss += criterion(out, labels).item()
            correct += (out.argmax(1) == labels).sum().item()
    avg_val = total_loss / len(val_loader)
    val_acc = correct / len(val_ds)
    val_losses.append(avg_val)
    val_accs.append(val_acc)

    print(f"Epoch [{epoch:02d}/{EPOCHS}]  train={avg_train:.4f}  val={avg_val:.4f}  acc={val_acc:.4f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CLF_OUT / "best.pth")
        print(f"  checkpoint saved (acc={val_acc:.4f})")

torch.save(model.state_dict(), CLF_OUT / "last.pth")
print(f"\nDone. Best val accuracy: {best_val_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, label="Train loss")
axes[0].plot(val_losses,   label="Val loss")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(val_accs, label="Val accuracy", color="green")
axes[1].axhline(best_val_acc, linestyle="--", color="grey",
                label=f"Best: {best_val_acc:.4f}")
axes[1].set_title("Validation Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.suptitle("EfficientNet-B0 — Training on ACNE04 Patches", fontsize=13)
plt.tight_layout()
OUT_FIG = Path("outputs/figures")
OUT_FIG.mkdir(parents=True, exist_ok=True)
plt.savefig(OUT_FIG / "classifier_training_curves.png", dpi=150)
plt.show()
print("Saved -> outputs/figures/classifier_training_curves.png")

In [ ]:
# Download DermNet from Kaggle
import os
import shutil
from pathlib import Path

os.environ["KAGGLE_USERNAME"] = "your_username"   # <- paste yours
os.environ["KAGGLE_KEY"]      = "your_key"        # <- paste yours

!pip install -q kagglehub
import kagglehub

path = kagglehub.dataset_download("shubhamgoel27/dermnet")
print("Downloaded to:", path)

DERMNET_DIR = Path("data/dermnet")
DERMNET_DIR.mkdir(parents=True, exist_ok=True)
shutil.copytree(path, str(DERMNET_DIR), dirs_exist_ok=True)
print(f"DermNet ready at {DERMNET_DIR}")

---
## Section 4 — DermNet Evaluation

Evaluates the trained classifier on the DermNet test set.

**Class mapping:** `Acne and Rosacea Photos` → acne (1), all other 22 conditions → non-acne (0)

**Note on class imbalance:** DermNet test set is 312 acne vs 3,690 non-acne (8% vs 92%).
A naive model predicting non-acne always would score 92% accuracy.
F1 (acne class) and AUROC are the meaningful metrics here.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
from pathlib import Path
from PIL import Image

DERMNET_DIR = Path("data/dermnet")
CLF_OUT     = Path("outputs/classifier")
ACNE_FOLDER = "Acne and Rosacea Photos"
CONF        = 0.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
class DermNetBinary(Dataset):
    """DermNet test set with binary acne / non-acne labels."""
    def __init__(self, split, transform):
        self.samples = []
        root = DERMNET_DIR / split
        for folder in sorted(root.iterdir()):
            label = 1 if folder.name == ACNE_FOLDER else 0
            for img_path in sorted(folder.glob("*")):
                if img_path.suffix.lower() in (".jpg", ".jpeg", ".png"):
                    self.samples.append((img_path, label))
        self.transform = transform

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label

test_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_ds     = DermNetBinary("test", test_tf)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False,
                         num_workers=4, pin_memory=True)
print(f"Test set: {len(test_ds)} images")
acne_count = sum(1 for _, l in test_ds.samples if l == 1)
print(f"  acne={acne_count}  non_acne={len(test_ds)-acne_count}")

In [ ]:
# Load best checkpoint
model = models.efficientnet_b0(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(model.classifier[1].in_features, 2),
)
model.load_state_dict(torch.load(str(CLF_OUT / "best.pth"),
                                  map_location=device, weights_only=False))
model.to(device).eval()
print("Model loaded.")

In [ ]:
# Run inference
all_probs, all_preds, all_labels = [], [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        probs = torch.softmax(model(imgs), dim=1)[:, 1]  # prob of acne
        preds = (probs >= CONF).long()
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_probs  = np.array(all_probs)
all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc   = accuracy_score(all_labels, all_preds)
f1    = f1_score(all_labels, all_preds, pos_label=1, zero_division=0)
auroc = roc_auc_score(all_labels, all_probs)

print(f"Accuracy : {acc:.4f}  (naive baseline: {1 - acne_count/len(test_ds):.4f})")
print(f"F1 (acne): {f1:.4f}")
print(f"AUROC    : {auroc:.4f}")

In [ ]:
# Flipped baseline: use (1 - prob) as acne score
# AUROC < 0.5 means predictions are anti-correlated with ground truth.
# Flipping recovers 1 - AUROC without any retraining.
# Requires knowing the inversion exists (detectable from >=1 labeled DermNet sample).

flipped_probs = 1 - all_probs
flipped_preds = (flipped_probs >= 0.5).astype(int)

flip_acc   = accuracy_score(all_labels, flipped_preds)
flip_f1    = f1_score(all_labels, flipped_preds, pos_label=1, zero_division=0)
flip_auroc = roc_auc_score(all_labels, flipped_probs)

print(f"Flipped baseline → Acc={flip_acc:.4f}  F1={flip_f1:.4f}  AUROC={flip_auroc:.4f}")
print(f"(vs baseline     → Acc={acc:.4f}  F1={f1:.4f}  AUROC={auroc:.4f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix — layout: rows=Predicted, cols=Actually, positive (acne) first
# sklearn cm: rows=actual, cols=predicted → transpose + flip for target layout
cm = confusion_matrix(all_labels, all_preds)
cm_display = cm.T[::-1, ::-1]   # rows=predicted, cols=actual, acne first
disp = ConfusionMatrixDisplay(cm_display, display_labels=["acne", "non-acne"])
disp.plot(ax=axes[0], colorbar=False)
axes[0].set_xlabel("Actually")
axes[0].set_ylabel("Predicted")
axes[0].set_title("Confusion Matrix — DermNet Test Set")

# ROC curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
axes[1].plot(fpr, tpr, label=f"AUROC = {auroc:.4f}")
axes[1].plot([0,1],[0,1], "k--", label="Random")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve — DermNet Test Set")
axes[1].legend()

plt.tight_layout()
OUT_FIG = Path("outputs/figures")
OUT_FIG.mkdir(parents=True, exist_ok=True)
plt.savefig(OUT_FIG / "dermnet_evaluation.png", dpi=150)
plt.show()

import json as _json
with open("outputs/dermnet_results.json", "w") as f:
    _json.dump({"accuracy": acc, "f1_acne": f1, "auroc": auroc}, f, indent=2)
print("Saved -> outputs/dermnet_results.json")

---
## Section 5 — Comprehensive Domain Adaptation

Seven techniques applied to bridge the ACNE04 → DermNet domain gap:

| # | Technique | Why it helps |
|---|---|---|
| 1 | **Reinhard color normalization** (LAB space) | Match ACNE04 color/lighting statistics |
| 2 | **Histogram matching** | Align pixel distribution to ACNE04 reference |
| 3 | **Freeze backbone → head-only fine-tune** | 20 samples can't safely update 5M params |
| 4 | **Layer-wise LR** (unfreeze last feature block) | Gradual domain adaptation without destroying ImageNet features |
| 5 | **Threshold optimization** on 20 labeled samples | Directly tune CONF for F1 on target domain |
| 6 | **6-crop TTA** (FiveCrop + horizontal flip) | Free accuracy from multi-view averaging |
| 7 | **FaceNet VGGFace2 backbone** | Face-specific features vs generic ImageNet features |

**Note on CycleGAN:** Full CycleGAN training (ACNE04 ↔ DermNet style transfer) requires 4–8 hrs on A100. We implement Reinhard + histogram matching as a practical equivalent (same goal: make DermNet images look like ACNE04). The bonus section below shows AdaIN-style feature-level transfer as a lightweight CycleGAN proxy.
**Dependencies:** Sections 3 (trained model: `outputs/classifier/best.pth`) and 4 (DermNet test set in memory: `model`, `test_ds`, `acc`, `f1`, `auroc`) must run first.

In [ ]:
import random, json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# scikit-image ships with Colab — no extra install needed
from skimage import color as skcolor
from skimage.exposure import match_histograms

%matplotlib inline

DERMNET_DIR = Path('data/dermnet')
PATCH_DIR   = Path('data/patches')
CLF_OUT     = Path('outputs/classifier')
ACNE_FOLDER = 'Acne and Rosacea Photos'
OUT_FIG     = Path('outputs/figures')
OUT_FIG.mkdir(parents=True, exist_ok=True)

# Standard eval transform (no augmentation)
_norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
base_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(), _norm,
])

# ── Sample 20 DermNet training images ────────────────────────────────────────
random.seed(42)
acne_files = [f for f in (DERMNET_DIR / 'train' / ACNE_FOLDER).glob('*')
              if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
non_acne_files = []
for folder in (DERMNET_DIR / 'train').iterdir():
    if folder.name != ACNE_FOLDER:
        non_acne_files += [f for f in folder.glob('*')
                           if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]

few_shot = ([(f, 1) for f in random.sample(acne_files, 10)] +
            [(f, 0) for f in random.sample(non_acne_files, 10)])
print(f'Few-shot set: {len(few_shot)} samples (10 acne + 10 non-acne)')

# ── Compute ACNE04 LAB statistics for Reinhard normalization ─────────────────
acne04_paths = list((PATCH_DIR / 'train' / 'acne').glob('*.jpg'))
sample_paths = random.sample(acne04_paths, min(400, len(acne04_paths)))

pixels = []
for p in sample_paths:
    arr = np.array(Image.open(p).convert('RGB').resize((224, 224))) / 255.0
    pixels.append(skcolor.rgb2lab(arr).reshape(-1, 3))

acne04_pixels   = np.concatenate(pixels, axis=0)
ACNE04_LAB_MEAN = acne04_pixels.mean(axis=0)
ACNE04_LAB_STD  = acne04_pixels.std(axis=0)
ACNE04_REF_IMG  = Image.open(random.choice(acne04_paths)).convert('RGB').resize((224, 224))

print(f'ACNE04 LAB mean: {ACNE04_LAB_MEAN.round(3)}')
print(f'ACNE04 LAB std : {ACNE04_LAB_STD.round(3)}')
print('Setup complete.')

In [ ]:
# ── Technique 1 + 2: Reinhard color normalization + histogram matching ────────

def reinhard_normalize(img_pil):
    """Transfer ACNE04 LAB color statistics to a DermNet image (Reinhard et al. 2001)."""
    arr = np.array(img_pil.convert('RGB').resize((224, 224))) / 255.0
    lab = skcolor.rgb2lab(arr)
    src_mean = lab.reshape(-1, 3).mean(axis=0)
    src_std  = lab.reshape(-1, 3).std(axis=0) + 1e-6
    for c in range(3):
        lab[:, :, c] = ((lab[:, :, c] - src_mean[c]) / src_std[c]
                        * ACNE04_LAB_STD[c] + ACNE04_LAB_MEAN[c])
    lab[:, :, 0] = np.clip(lab[:, :, 0], 0, 100)
    lab[:, :, 1] = np.clip(lab[:, :, 1], -128, 127)
    lab[:, :, 2] = np.clip(lab[:, :, 2], -128, 127)
    rgb = np.clip(skcolor.lab2rgb(lab), 0, 1)
    return Image.fromarray((rgb * 255).astype(np.uint8))

def histo_match(img_pil):
    """Match DermNet histogram to ACNE04 reference image per channel."""
    arr     = np.array(img_pil.convert('RGB').resize((224, 224)))
    matched = match_histograms(arr, np.array(ACNE04_REF_IMG), channel_axis=-1)
    return Image.fromarray(matched.astype(np.uint8))

def preprocess_dermnet(img_pil):
    """Reinhard normalization → histogram matching. Applied to all DermNet images."""
    return histo_match(reinhard_normalize(img_pil))

# Visualize the pipeline on a sample DermNet acne image
sample_img = Image.open(random.choice(acne_files)).convert('RGB')
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, title in zip(axes,
    [sample_img, ACNE04_REF_IMG, reinhard_normalize(sample_img), preprocess_dermnet(sample_img)],
    ['Original DermNet', 'ACNE04 reference', '+ Reinhard norm', '+ Histo match']):
    ax.imshow(img.resize((224, 224)) if img.size != (224,224) else img)
    ax.set_title(title, fontsize=9); ax.axis('off')

plt.suptitle('Color Normalization Pipeline — DermNet → ACNE04 color space', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_FIG / 'color_normalization.png', dpi=150)
plt.show()
print('Saved -> outputs/figures/color_normalization.png')

In [ ]:
# Ablation point: original model + color norm only (no fine-tuning)
# Isolates the contribution of preprocessing from fine-tuning
print('Ablation: original model (best.pth) + color normalization...')

model.eval()
cn_probs, cn_labels_raw = [], []

with torch.no_grad():
    for path, label in test_ds.samples:
        img = preprocess_dermnet(Image.open(path).convert('RGB'))
        t   = base_tf(img).unsqueeze(0).to(device)
        p   = torch.softmax(model(t), dim=1)[0, 1].item()
        cn_probs.append(p)
        cn_labels_raw.append(label)

cn_probs  = np.array(cn_probs)
cn_labels_raw = np.array(cn_labels_raw)
cn_preds  = (cn_probs >= 0.5).astype(int)
cn_acc    = accuracy_score(cn_labels_raw, cn_preds)
cn_f1     = f1_score(cn_labels_raw, cn_preds, pos_label=1, zero_division=0)
cn_auroc  = roc_auc_score(cn_labels_raw, cn_probs)
print(f'Color norm only → Acc={cn_acc:.4f}  F1={cn_f1:.4f}  AUROC={cn_auroc:.4f}')
print(f'(Baseline was  → Acc={acc:.4f}  F1={f1:.4f}  AUROC={auroc:.4f})')

In [ ]:
# ── Techniques 3 + 4: 2-stage fine-tuning ────────────────────────────────────
# Stage 1: Freeze backbone — train head only with preprocessed DermNet samples
# Stage 2: Unfreeze last feature block with 100x lower LR (layer-wise LR)

class FewShotDataset(Dataset):
    def __init__(self, samples, transform, preprocess_fn=None):
        self.samples = samples
        self.transform = transform
        self.preprocess_fn = preprocess_fn
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.preprocess_fn:
            img = self.preprocess_fn(img)
        return self.transform(img), label

ft_aug_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(), _norm,
])

ft_loader = DataLoader(
    FewShotDataset(few_shot, ft_aug_tf, preprocess_fn=preprocess_dermnet),
    batch_size=4, shuffle=True
)

# Load base checkpoint fresh
ft_model = models.efficientnet_b0(weights=None)
ft_model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(ft_model.classifier[1].in_features, 2),
)
ft_model.load_state_dict(torch.load(str(CLF_OUT / 'best.pth'),
                                     map_location=device, weights_only=False))
ft_model.to(device)
criterion = nn.CrossEntropyLoss()

# ── Stage 1: head only ───────────────────────────────────────────────────────
print('Stage 1: head-only (50 epochs, lr=5e-3)...')
for p in ft_model.features.parameters():
    p.requires_grad = False

opt1 = torch.optim.Adam(ft_model.classifier.parameters(), lr=5e-3, weight_decay=1e-3)
sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=50)

ft_model.train()
for epoch in range(1, 51):
    for imgs, labels in ft_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        opt1.zero_grad()
        loss = criterion(ft_model(imgs), labels)
        loss.backward(); opt1.step()
    sch1.step()
    if epoch % 10 == 0:
        print(f'  Epoch {epoch:02d}/50  loss={loss.item():.4f}')

# ── Stage 2: unfreeze last block with 100x lower LR ─────────────────────────
print('Stage 2: layer-wise LR — last feature block (30 epochs)...')
for p in ft_model.features[-1].parameters():
    p.requires_grad = True

opt2 = torch.optim.Adam([
    {'params': ft_model.features[-1].parameters(), 'lr': 5e-6},
    {'params': ft_model.classifier.parameters(),   'lr': 5e-5},
], weight_decay=1e-4)
sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=30)

for epoch in range(1, 31):
    ft_model.train()
    for imgs, labels in ft_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        opt2.zero_grad()
        loss = criterion(ft_model(imgs), labels)
        loss.backward(); opt2.step()
    sch2.step()
    if epoch % 10 == 0:
        print(f'  Epoch {epoch:02d}/30  loss={loss.item():.4f}')

torch.save(ft_model.state_dict(), CLF_OUT / 'finetuned.pth')
print('\nSaved -> outputs/classifier/finetuned.pth')

In [ ]:
# ── Technique 5: Optimize decision threshold on 20 labeled training samples ──

ft_model.eval()
train_probs_th, train_labels_th = [], []

with torch.no_grad():
    for path, label in few_shot:
        img = preprocess_dermnet(Image.open(path).convert('RGB'))
        t   = base_tf(img).unsqueeze(0).to(device)
        p   = torch.softmax(ft_model(t), dim=1)[0, 1].item()
        train_probs_th.append(p)
        train_labels_th.append(label)

train_probs_th  = np.array(train_probs_th)
train_labels_th = np.array(train_labels_th)

best_t, best_f1_t = 0.5, 0.0
for t in np.linspace(0.05, 0.95, 91):
    preds = (train_probs_th >= t).astype(int)
    f = f1_score(train_labels_th, preds, pos_label=1, zero_division=0)
    if f > best_f1_t:
        best_f1_t, best_t = f, t

CONF = float(best_t)
print(f'Optimal threshold : {CONF:.2f}  (F1={best_f1_t:.4f} on 20 training samples)')
print(f'Default was 0.50  → tuned to {CONF:.2f}')

In [ ]:
# ── Technique 6: 6-crop TTA (FiveCrop + horizontal flip) ─────────────────────

def tta_predict(model, img_preprocessed, device):
    """Run 6 deterministic crops through the model and average probabilities."""
    img_256 = img_preprocessed.resize((256, 256), Image.BILINEAR)
    crops   = list(transforms.FiveCrop(224)(img_256))          # 5 PIL images
    crops.append(crops[4].transpose(Image.FLIP_LEFT_RIGHT))    # + center flip
    batch = torch.stack([_norm(transforms.ToTensor()(c)) for c in crops]).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(batch), dim=1)[:, 1]
    return probs.mean().item()

def eval_full_pipeline(model, samples, device, threshold=0.5):
    """Evaluate: preprocess_dermnet + 6-crop TTA + optimal threshold."""
    model.eval()
    all_probs, all_labels = [], []
    for path, label in samples:
        img  = preprocess_dermnet(Image.open(path).convert('RGB'))
        prob = tta_predict(model, img, device)
        all_probs.append(prob)
        all_labels.append(label)
    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_preds  = (all_probs >= threshold).astype(int)
    return (all_probs, all_preds, all_labels,
            accuracy_score(all_labels, all_preds),
            f1_score(all_labels, all_preds, pos_label=1, zero_division=0),
            roc_auc_score(all_labels, all_probs))

print('Evaluating full pipeline (color norm + 2-stage FT + TTA + tuned threshold)...')
print('~2-3 min on A100 (6 forward passes per image × 4002 images)')

ft_probs, ft_preds, ft_labels, ft_acc, ft_f1, ft_auroc = eval_full_pipeline(
    ft_model, test_ds.samples, device, threshold=CONF
)
print(f'\nEfficientNet full pipeline → Acc={ft_acc:.4f}  F1={ft_f1:.4f}  AUROC={ft_auroc:.4f}')

In [ ]:
# ── Technique 7: FaceNet VGGFace2 pretrained backbone ────────────────────────
!pip install -q facenet-pytorch

from facenet_pytorch import InceptionResnetV1

class FaceAcneClassifier(nn.Module):
    """InceptionResnetV1 pretrained on VGGFace2, head fine-tuned on 20 DermNet samples."""
    def __init__(self):
        super().__init__()
        self.backbone = InceptionResnetV1(pretrained='vggface2')
        for p in self.backbone.parameters():
            p.requires_grad = False          # freeze all backbone weights
        self.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(512, 2))
    def forward(self, x):
        return self.classifier(self.backbone(x))

facenet_model = FaceAcneClassifier().to(device)
trainable_k   = sum(p.numel() for p in facenet_model.parameters() if p.requires_grad) / 1e3
print(f'FaceNet (VGGFace2) loaded — {trainable_k:.1f}K trainable params (head only)')

# FaceNet expects 160×160 images normalized to [-1, 1]
facenet_tf = transforms.Compose([
    transforms.Resize(182),
    transforms.CenterCrop(160),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

fn_loader = DataLoader(
    FewShotDataset(few_shot, facenet_tf, preprocess_fn=preprocess_dermnet),
    batch_size=4, shuffle=True
)

fn_optim = torch.optim.Adam(facenet_model.classifier.parameters(), lr=1e-2, weight_decay=1e-3)
fn_sched = torch.optim.lr_scheduler.CosineAnnealingLR(fn_optim, T_max=60)

print('Fine-tuning FaceNet head (60 epochs)...')
fn_crit = nn.CrossEntropyLoss()
facenet_model.train()
for epoch in range(1, 61):
    for imgs, labels in fn_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        fn_optim.zero_grad()
        loss = fn_crit(facenet_model(imgs), labels)
        loss.backward(); fn_optim.step()
    fn_sched.step()
    if epoch % 15 == 0:
        print(f'  Epoch {epoch:02d}/60  loss={loss.item():.4f}')

# Threshold-tune FaceNet on the 20 training samples
facenet_model.eval()
fn_train_probs, fn_train_labels = [], []
with torch.no_grad():
    for path, label in few_shot:
        img = preprocess_dermnet(Image.open(path).convert('RGB'))
        t   = facenet_tf(img).unsqueeze(0).to(device)
        p   = torch.softmax(facenet_model(t), dim=1)[0, 1].item()
        fn_train_probs.append(p)
        fn_train_labels.append(label)

fn_train_probs  = np.array(fn_train_probs)
fn_train_labels = np.array(fn_train_labels)
fn_best_t, fn_best_f1 = 0.5, 0.0
for t in np.linspace(0.05, 0.95, 91):
    preds = (fn_train_probs >= t).astype(int)
    f = f1_score(fn_train_labels, preds, pos_label=1, zero_division=0)
    if f > fn_best_f1:
        fn_best_f1, fn_best_t = f, t
print(f'FaceNet optimal threshold: {fn_best_t:.2f}')

# Evaluate on full test set (no TTA — backbone already produces stable 512-d embeddings)
fn_probs_all, fn_labels_all = [], []
with torch.no_grad():
    for path, label in test_ds.samples:
        img = preprocess_dermnet(Image.open(path).convert('RGB'))
        t   = facenet_tf(img).unsqueeze(0).to(device)
        p   = torch.softmax(facenet_model(t), dim=1)[0, 1].item()
        fn_probs_all.append(p)
        fn_labels_all.append(label)

fn_probs_all  = np.array(fn_probs_all)
fn_labels_all = np.array(fn_labels_all)
fn_preds_all  = (fn_probs_all >= fn_best_t).astype(int)
fn_acc   = accuracy_score(fn_labels_all, fn_preds_all)
fn_f1    = f1_score(fn_labels_all, fn_preds_all, pos_label=1, zero_division=0)
fn_auroc = roc_auc_score(fn_labels_all, fn_probs_all)
print(f'FaceNet → Acc={fn_acc:.4f}  F1={fn_f1:.4f}  AUROC={fn_auroc:.4f}')

In [ ]:
# ── Full ablation table ───────────────────────────────────────────────────────

naive_acc = 1 - sum(l == 1 for _, l in test_ds.samples) / len(test_ds.samples)

rows = [
    ('Naive (predict all non-acne)',               naive_acc, 0.0,      None),
    ('EfficientNet-B0 (no adaptation)',             acc,       f1,       auroc),
    ('+ Flip score (1 - prob)',                     flip_acc,  flip_f1,  flip_auroc),
    ('+ Reinhard + histo-match only',               cn_acc,    cn_f1,    cn_auroc),
    ('+ 2-stage FT + color norm + thresh + TTA',   ft_acc,    ft_f1,    ft_auroc),
    ('FaceNet VGGFace2 + color norm + threshold',  fn_acc,    fn_f1,    fn_auroc),
]

print(f'{"Strategy":<52} {"Accuracy":>9} {"F1(acne)":>9} {"AUROC":>7}')
print('─' * 81)
for name, a, f, au in rows:
    auroc_str = f'{au:.4f}' if au is not None else '  —  '
    print(f'{name:<52} {a:>9.4f} {f:>9.4f} {auroc_str:>7}')

print()
best_auroc = max(r[3] for r in rows if r[3] is not None)
print(f'Best AUROC: {best_auroc:.4f}')

with open('outputs/dermnet_results.json', 'w') as fp:
    json.dump({
        'naive_baseline':    {'accuracy': naive_acc},
        'baseline':          {'accuracy': acc,       'f1_acne': f1,       'auroc': auroc},
        'baseline_flipped':  {'accuracy': flip_acc,  'f1_acne': flip_f1,  'auroc': flip_auroc,
                              'note': '1 - baseline prob; requires knowing inversion exists'},
        'color_norm_only':   {'accuracy': cn_acc,    'f1_acne': cn_f1,    'auroc': cn_auroc},
        'efficientnet_full': {'accuracy': ft_acc,    'f1_acne': ft_f1,    'auroc': ft_auroc,
                              'threshold': CONF},
        'facenet':           {'accuracy': fn_acc,    'f1_acne': fn_f1,    'auroc': fn_auroc,
                              'threshold': fn_best_t},
    }, fp, indent=2)
print('Saved -> outputs/dermnet_results.json')

In [ ]:
# ── Final model vs baseline: confusion matrix + ROC side-by-side ──────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

def plot_cm(ax, labels, preds, title):
    cm = confusion_matrix(labels, preds)
    cm_display = cm.T[::-1, ::-1]
    disp = ConfusionMatrixDisplay(cm_display, display_labels=['acne', 'non-acne'])
    disp.plot(ax=ax, colorbar=False)
    ax.set_xlabel('Actually'); ax.set_ylabel('Predicted')
    ax.set_title(title)

def plot_roc(ax, labels, probs, auroc, title):
    fpr, tpr, _ = roc_curve(labels, probs)
    ax.plot(fpr, tpr, label=f'AUROC = {auroc:.4f}')
    ax.plot([0, 1], [0, 1], 'k--', label='Random')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(title); ax.legend()

# Baseline (threshold=0.5)
baseline_preds = (all_probs >= 0.5).astype(int)
plot_cm(axes[0, 0], all_labels, baseline_preds,  'Confusion Matrix — Baseline (no adapt)')
plot_roc(axes[1, 0], all_labels, all_probs, auroc, 'ROC Curve — Baseline')

# Full pipeline (color norm + 2-stage FT + TTA + threshold=0.07)
plot_cm(axes[0, 1], ft_labels, ft_preds,  f'Confusion Matrix — Full Pipeline (thresh={CONF})')
plot_roc(axes[1, 1], ft_labels, ft_probs, ft_auroc, 'ROC Curve — Full Pipeline')

plt.suptitle('Baseline vs Full Pipeline — DermNet Test Set (4,002 images)', fontsize=14)
plt.tight_layout()

OUT_PART2 = Path('outputs/part2')
OUT_PART2.mkdir(parents=True, exist_ok=True)
plt.savefig(OUT_PART2 / 'final_model_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> outputs/part2/final_model_results.png')

---
## Section 6 — Domain Adaptation

ACNE04 (face selfies) and DermNet (clinical photos) differ in lighting,
color profile, and scale. We bridge this gap through augmentation baked
into training:

| Technique | Purpose |
|---|---|
| `ColorJitter(brightness, contrast, saturation, hue)` | Robust to lighting differences |
| `RandomResizedCrop(scale=0.7-1.0)` | Robust to scale/zoom variation |
| `GaussianBlur` | Robust to sharpness differences |
| `RandomHorizontalFlip` | Left/right invariance |

This section visualises the domain gap and the effect of augmentation.

In [ ]:
import random
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

PATCH_DIR   = Path("data/patches")
ACNE_FOLDER = "Acne and Rosacea Photos"
random.seed(42)

acne04_samples  = random.sample(list((PATCH_DIR / "train" / "acne").glob("*.jpg")), 4)
dermnet_samples = random.sample(list((DERMNET_DIR / "train" / ACNE_FOLDER).glob("*")), 4)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col, f in enumerate(acne04_samples):
    axes[0][col].imshow(Image.open(f))
    axes[0][col].set_title("ACNE04 patch", fontsize=9)
    axes[0][col].axis("off")
for col, f in enumerate(dermnet_samples):
    axes[1][col].imshow(Image.open(f))
    axes[1][col].set_title("DermNet acne", fontsize=9)
    axes[1][col].axis("off")

plt.suptitle("Domain Gap — ACNE04 (top) vs DermNet (bottom)", fontsize=13)
plt.tight_layout()
OUT_FIG = Path("outputs/figures")
OUT_FIG.mkdir(parents=True, exist_ok=True)
plt.savefig(OUT_FIG / "domain_gap.png", dpi=150)
plt.show()
print("Saved -> outputs/figures/domain_gap.png")

In [ ]:
# Show augmentation applied to same patch 5 ways
aug_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
])

src_img = Image.open(acne04_samples[0]).convert("RGB")

fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(src_img)
axes[0].set_title("Original", fontsize=9); axes[0].axis("off")
for i in range(1, 6):
    axes[i].imshow(aug_tf(src_img))
    axes[i].set_title(f"Augmented {i}", fontsize=9); axes[i].axis("off")

plt.suptitle("Training Augmentation — same patch, 5 random transforms", fontsize=12)
plt.tight_layout()
plt.savefig(OUT_FIG / "augmentation_examples.png", dpi=150)
plt.show()
print("Saved -> outputs/figures/augmentation_examples.png")

In [ ]:
# RGB channel statistics — quantify domain gap
import numpy as np

def channel_stats(paths, n=200):
    sample = random.sample(list(paths), min(n, len(list(paths))))
    means = [np.array(Image.open(p).convert("RGB").resize((224,224)),
                      dtype=float).mean(axis=(0,1)) / 255.0
             for p in sample]
    means = np.array(means)
    return means.mean(axis=0), means.std(axis=0)

a_mean, a_std = channel_stats(list((PATCH_DIR / "train" / "acne").glob("*.jpg")))
d_mean, d_std = channel_stats(list((DERMNET_DIR / "train" / ACNE_FOLDER).glob("*")))

x = np.arange(3)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - 0.2, a_mean, 0.35, yerr=a_std, label="ACNE04", color="#FF6B6B", capsize=4)
ax.bar(x + 0.2, d_mean, 0.35, yerr=d_std, label="DermNet", color="#4ECDC4", capsize=4)
ax.set_xticks(x); ax.set_xticklabels(["R", "G", "B"])
ax.set_ylabel("Mean pixel value (0-1)")
ax.set_title("RGB Channel Statistics — ACNE04 vs DermNet")
ax.legend(); plt.tight_layout()
plt.savefig(OUT_FIG / "channel_stats.png", dpi=150)
plt.show()
print(f"ACNE04  mean RGB: {a_mean.round(3)}")
print(f"DermNet mean RGB: {d_mean.round(3)}")
print(f"Difference      : {(d_mean - a_mean).round(3)}")

---
## Section 7 — Grad-CAM Visualizations

Shows **where** the fine-tuned model looks when classifying DermNet images.

- Target layer: `model.features[-1]` (last EfficientNet-B0 conv block)
- 10 images selected: 5 acne + 5 non-acne from the test set
- Green title = correct prediction, red = wrong

In [ ]:
import sys
!pip install opencv-contrib-python
# Restart the kernel to ensure the new cv2 module is loaded
if 'cv2' in sys.modules:
    del sys.modules['cv2']

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import numpy as np, random
import matplotlib.pyplot as plt
from PIL import Image

# ft_model, test_ds, test_tf, device, CONF defined in Sections 4 & 5
ft_model.eval()

random.seed(99)
acne_idx    = [i for i,(_, l) in enumerate(test_ds.samples) if l == 1]
nonacne_idx = [i for i,(_, l) in enumerate(test_ds.samples) if l == 0]
selected    = random.sample(acne_idx, 5) + random.sample(nonacne_idx, 5)

target_layers = [ft_model.features[-1]]

fig, axes = plt.subplots(2, 5, figsize=(18, 8))

with GradCAM(model=ft_model, target_layers=target_layers) as cam_engine:
    for plot_i, idx in enumerate(selected):
        row, col = divmod(plot_i, 5)
        img_path, true_label = test_ds.samples[idx]

        img_pil = Image.open(img_path).convert('RGB')
        tensor  = test_tf(img_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            prob = torch.softmax(ft_model(tensor), dim=1)[0, 1].item()
        pred = 1 if prob >= CONF else 0

        grayscale_cam = cam_engine(
            input_tensor=tensor,
            targets=[ClassifierOutputTarget(pred)]
        )[0]

        raw = np.array(img_pil.resize((224, 224), Image.BILINEAR), dtype=np.float32) / 255.0
        cam_img = show_cam_on_image(raw, grayscale_cam, use_rgb=True)

        true_str = 'acne' if true_label == 1 else 'non-acne'
        pred_str = 'acne' if pred == 1 else 'non-acne'
        color    = 'green' if true_label == pred else 'red'

        axes[row, col].imshow(cam_img)
        axes[row, col].set_title(
            f'True: {true_str}\nPred: {pred_str} ({prob:.2f})',
            fontsize=8, color=color
        )
        axes[row, col].axis('off')

plt.suptitle(
    'Grad-CAM — Fine-tuned EfficientNet-B0 on DermNet Test\n'
    'Row 1: acne samples  |  Row 2: non-acne samples  (green=correct, red=wrong)',
    fontsize=12
)
plt.tight_layout()
OUT_FIG = Path('outputs/figures')
OUT_FIG.mkdir(parents=True, exist_ok=True)
plt.savefig(OUT_FIG / 'gradcam_dermnet.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/gradcam_dermnet.png')